# 三圆面积和RTK真值误差

In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import math
import csv
import io
import traceback
from pathlib import Path
from typing import Tuple, Optional, Dict, Any, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. Matplotlib config
# =========================================================
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# =========================================================
# 1. Paths and parameters
# =========================================================
DATA_PAIRS = [
    {
        "name": "1试验",
        "uwb": Path(r"D:\Desktop\missing_output\D1_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\1.txt"),
    },
    {
        "name": "2圆形",
        "uwb": Path(r"D:\Desktop\missing_output\D2_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\2.txt"),
    },
    {
        "name": "3方形",
        "uwb": Path(r"D:\Desktop\missing_output\D3_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\3.txt"),
    },
    {
        "name": "4凹凸",
        "uwb": Path(r"D:\Desktop\missing_output\D4_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\4.txt"),
    },
]

OUTPUT_DIR = Path(r"D:\Desktop\missing_output\batch_output")
OUTPUT_SUMMARY_XLSX = OUTPUT_DIR / "uwb_rtk_batch_summary.xlsx"
OUTPUT_SCENE_INFO_XLSX = OUTPUT_DIR / "parking_scene_reference.xlsx"

ANCHORS_2D = {
    0: (0.00, 0.00),
    1: (14.95, 0.00),
    2: (18.51, 33.15),
}

UWB_FLIP_X = True

TIME_SHIFT_CANDIDATES_COARSE = np.arange(-10.0, 10.01, 0.2)
TIME_SHIFT_FINE_STEP = 0.02
TIME_SHIFT_FINE_HALF_WIDTH = 0.3

FIG_DPI = 220

MAX_SPEED_MPS = 3.0
PREDICTION_BLEND = 0.80
MIN_DT_FOR_VEL = 1e-6

SMOOTH_WIN = 5
SMOOTH_ENABLE = True

MAX_CROSS_RESIDUAL_FOR_ALIGNMENT = 1.0
MIN_ALIGNMENT_POINTS = 8
MAX_CROSS_RESIDUAL_FOR_ALIGNMENT_RELAXED = 2.5
MIN_ALIGNMENT_POINTS_RELAXED = 5

MAX_GAP_FILL_POINTS = 8

# =========================================================
# 2. Logging helpers
# =========================================================
def ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def append_log(log_path: Path, text: str) -> None:
    ensure_parent_dir(log_path)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(text.rstrip() + "\n")

def log_print(log_path: Path, text: str) -> None:
    print(text)
    append_log(log_path, text)

def to_num_series(s):
    return pd.to_numeric(s, errors="coerce")

def infer_rate(t: np.ndarray) -> float:
    t = np.asarray(t, dtype=float)
    if len(t) < 3:
        return float("nan")
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return float("nan")
    return 1.0 / float(np.median(dt))

def finite_count_2cols(a, b) -> int:
    a1 = pd.to_numeric(pd.Series(a), errors="coerce")
    b1 = pd.to_numeric(pd.Series(b), errors="coerce")
    return int((a1.notna() & b1.notna()).sum())

def interp1_nan_safe(t_ref, v_ref, t_query):
    t_ref = np.asarray(t_ref, dtype=float)
    v_ref = np.asarray(v_ref, dtype=float)
    t_query = np.asarray(t_query, dtype=float)

    m = np.isfinite(t_ref) & np.isfinite(v_ref)
    if np.count_nonzero(m) < 2:
        return np.full_like(t_query, np.nan, dtype=float)

    t0 = t_ref[m]
    v0 = v_ref[m]
    order = np.argsort(t0)
    t0 = t0[order]
    v0 = v0[order]

    out = np.interp(t_query, t0, v0)
    out[(t_query < t0[0]) | (t_query > t0[-1])] = np.nan
    return out

def interp2_at_times(t_ref, x_ref, y_ref, t_query):
    xi = interp1_nan_safe(t_ref, x_ref, t_query)
    yi = interp1_nan_safe(t_ref, y_ref, t_query)
    return xi, yi

def moving_average_1d_preserve_nan(x: np.ndarray, window: int = 5) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if window < 2:
        return x.copy()
    if window % 2 == 0:
        window += 1

    y = x.copy()
    pad = window // 2
    for i in range(len(x)):
        lo = max(0, i - pad)
        hi = min(len(x), i + pad + 1)
        seg = x[lo:hi]
        if np.count_nonzero(np.isfinite(seg)) >= 1:
            y[i] = np.nanmean(seg)
        else:
            y[i] = np.nan
    return y

def smooth_xy(x: np.ndarray, y: np.ndarray, window: int = 5) -> Tuple[np.ndarray, np.ndarray]:
    return moving_average_1d_preserve_nan(x, window), moving_average_1d_preserve_nan(y, window)

def calc_rmse(x1, y1, x2, y2):
    x1 = np.asarray(x1, dtype=float)
    y1 = np.asarray(y1, dtype=float)
    x2 = np.asarray(x2, dtype=float)
    y2 = np.asarray(y2, dtype=float)

    m = np.isfinite(x1) & np.isfinite(y1) & np.isfinite(x2) & np.isfinite(y2)
    if np.count_nonzero(m) < 3:
        return np.inf
    err = np.sqrt((x1[m] - x2[m]) ** 2 + (y1[m] - y2[m]) ** 2)
    return float(np.sqrt(np.mean(err ** 2)))

def is_valid_number(v) -> bool:
    try:
        return bool(np.isfinite(float(v)))
    except Exception:
        return False

def is_valid_xy(x, y) -> bool:
    return is_valid_number(x) and is_valid_number(y)

# =========================================================
# 3. Load UWB CSV/XLSX
# =========================================================
def read_uwb_csv_with_trailing_empty_fields(file_path: Path) -> pd.DataFrame:
    """
    安全读取 UWB CSV：
    1. 解决表头末尾空列与数据行末尾多余逗号导致的时间列错位；
    2. 只移除末尾“多出来的空字段”，不删除中间缺测值；
    3. 多出的字段只要包含非空数据就报错，防止静默丢失；
    4. 保留 pandas 原有的数值类型推断与缺失值处理。
    本函数不修改磁盘上的原始 CSV。
    """
    # utf-8-sig 兼容普通 UTF-8 与 BOM；必要时兼容常见 Windows 中文编码。
    raw = file_path.read_bytes()
    try:
        csv_text = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        try:
            csv_text = raw.decode("gb18030")
        except UnicodeDecodeError as e:
            raise ValueError(f"UWB CSV 无法按 UTF-8 或 GB18030 解码: {file_path}") from e

    reader = csv.reader(io.StringIO(csv_text, newline=""))

    # 与 pandas 的 skip_blank_lines=True 行为一致：忽略开头的空白行。
    for header in reader:
        if header and any(field.strip() for field in header):
            break
    else:
        raise ValueError(f"UWB CSV 文件为空或没有有效表头: {file_path}")

    header = [field.strip() for field in header]
    removed_header_fields = 0
    while header and not header[-1]:
        header.pop()
        removed_header_fields += 1

    if not header:
        raise ValueError(f"UWB CSV 没有非空列名: {file_path}")

    cleaned = io.StringIO(newline="")
    writer = csv.writer(cleaned, lineterminator=chr(10))
    writer.writerow(header)

    corrected_rows = 0
    n_rows = 0
    for line_no, fields in enumerate(reader, start=2):
        if not fields:
            continue

        if len(fields) > len(header):
            extras = fields[len(header):]
            if any(value.strip() for value in extras):
                raise ValueError(
                    f"UWB CSV 第 {line_no} 行比表头多 {len(extras)} 个字段，"
                    "且其中存在非空数据；为防止丢失数据，已停止读取。"
                    "请检查原始文件的列名或分隔符。"
                )
            fields = fields[:len(header)]
            corrected_rows += 1
        elif len(fields) < len(header):
            # 末尾缺少字段时补空值；绝不向左/右移动已存在的字段。
            fields = fields + [""] * (len(header) - len(fields))

        writer.writerow(fields)
        n_rows += 1

    cleaned.seek(0)
    df = pd.read_csv(
        cleaned,
        index_col=False,  # 明确禁止把第一列时间戳当作索引
        na_values=["null", "NULL", "", "NaN", "nan"],
        keep_default_na=True,
    )
    if len(df) != n_rows:
        raise ValueError(
            f"CSV 解析后行数异常：原数据 {n_rows} 行，解析结果 {len(df)} 行。"
        )
    df.attrs["csv_removed_header_empty_fields"] = removed_header_fields
    df.attrs["csv_corrected_trailing_empty_rows"] = corrected_rows
    return df


def robust_load_uwb_csv_3ranges(file_path: Path) -> Tuple[np.ndarray, pd.DataFrame]:
    if not file_path.exists():
        raise FileNotFoundError(f"UWB input file not found: {file_path}")

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    else:
        df = read_uwb_csv_with_trailing_empty_fields(file_path)

    df.columns = [str(c).strip() for c in df.columns]

    time_col = None
    for c in ["rangetime(ms)", "rangetime", "time", "时间_s", "time_s", "t"]:
        if c in df.columns:
            time_col = c
            break
    if time_col is None:
        raise ValueError("未找到时间列，请至少包含 rangetime(ms) 或 时间_s。")

    required_cols = ["range0(m)", "range1(m)", "range2(m)"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"文件缺少必要列: {missing}")

    numeric_cols = [time_col, "range0(m)", "range1(m)", "range2(m)"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    t_raw = df[time_col].to_numpy(dtype=float)
    valid_count = int(np.isfinite(t_raw).sum())
    if valid_count < 2:
        raise ValueError(
            f"{time_col} 列有效数据不足（{valid_count}/{len(df)}）。"
            "请检查 CSV 的表头、行末分隔符及时间戳内容。"
        )

    first_valid = t_raw[np.isfinite(t_raw)][0]
    if time_col == "rangetime(ms)" or "ms" in time_col.lower():
        t = (t_raw - first_valid) / 1000.0
    else:
        t = t_raw - first_valid

    # 仅插值原本缺失的时间点，不改动任何已有的有效时间戳。
    idx = np.arange(len(t), dtype=float)
    m = np.isfinite(t)
    if np.count_nonzero(m) >= 2 and np.count_nonzero(~m) > 0:
        t[~m] = np.interp(idx[~m], idx[m], t[m])

    return t, df.copy()

# =========================================================
# 4. RTK parsing and global parking-lot scene system
# =========================================================
def nmea_degmin_to_deg(dm: str, hemi: str) -> float:
    if not dm:
        return float("nan")
    try:
        dm = float(dm)
    except Exception:
        return float("nan")

    deg = int(dm // 100)
    minutes = dm - deg * 100
    val = deg + minutes / 60.0
    if hemi in ("S", "W"):
        val = -val
    return val

def parse_gga_time_to_sec(tfield: str) -> Optional[float]:
    if not tfield:
        return None
    try:
        hh = int(tfield[0:2])
        mm = int(tfield[2:4])
        ss = float(tfield[4:])
        return hh * 3600 + mm * 60 + ss
    except Exception:
        return None

def make_time_monotonic(t_raw: np.ndarray) -> np.ndarray:
    t_raw = np.asarray(t_raw, dtype=float)
    if t_raw.size == 0:
        return t_raw.copy()

    t = np.array(t_raw, dtype=float)
    day_offset = 0.0
    prev = t[0]

    for i in range(1, len(t)):
        cur = t[i]
        while cur + day_offset < prev - 1.0:
            day_offset += 86400.0
        adj = cur + day_offset
        if adj < prev:
            adj = prev
        t[i] = adj
        prev = adj

    return t

def geodetic_to_local_xy(lat: float, lon: float, lat0: float, lon0: float) -> Tuple[float, float]:
    R = 6378137.0
    lat0r = math.radians(lat0)
    x = math.radians(lon - lon0) * math.cos(lat0r) * R
    y = math.radians(lat - lat0) * R
    return x, y

def parse_rtk_gga_raw(rtk_log_path: Path) -> pd.DataFrame:
    if not rtk_log_path.exists():
        raise FileNotFoundError(f"RTK file not found: {rtk_log_path}")

    times_raw = []
    lats_raw = []
    lons_raw = []

    with open(rtk_log_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not (line.startswith("$") and "GGA" in line):
                continue

            parts = line.split(",")
            if len(parts) < 7:
                continue

            fix = parts[6]
            if fix not in ("4", "5", "2", "1"):
                continue

            tsec = parse_gga_time_to_sec(parts[1] if len(parts) > 1 else "")
            lat = nmea_degmin_to_deg(parts[2], parts[3] if len(parts) > 3 else "")
            lon = nmea_degmin_to_deg(parts[4], parts[5] if len(parts) > 5 else "")

            if (tsec is None) or (not np.isfinite(lat)) or (not np.isfinite(lon)):
                continue

            times_raw.append(float(tsec))
            lats_raw.append(float(lat))
            lons_raw.append(float(lon))

    if len(times_raw) < 2:
        raise ValueError(f"RTK 文件有效 GGA 点不足: {rtk_log_path}")

    t = make_time_monotonic(np.array(times_raw, float))
    t = t - t[0]

    return pd.DataFrame({
        "时间_s": t,
        "纬度_deg": np.array(lats_raw, float),
        "经度_deg": np.array(lons_raw, float),
    })

def estimate_scene_rotation_from_points(x_all: np.ndarray, y_all: np.ndarray) -> float:
    pts = np.column_stack([x_all, y_all]).astype(float)
    m = np.isfinite(pts).all(axis=1)
    pts = pts[m]
    if len(pts) < 3:
        return 0.0

    pts0 = pts - pts.mean(axis=0, keepdims=True)
    cov = np.cov(pts0.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    v = eigvecs[:, np.argmax(eigvals)]  # 主方向

    theta = math.atan2(v[1], v[0])      # 主轴相对 East 的角度
    rot = -theta                        # 旋转到接近 X 正方向

    return float(rot)

def rotate_xy(x: np.ndarray, y: np.ndarray, theta_rad: float) -> Tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    c = math.cos(theta_rad)
    s = math.sin(theta_rad)
    xr = c * x - s * y
    yr = s * x + c * y
    return xr, yr

def build_global_parking_scene(data_pairs: List[Dict[str, Path]]) -> Dict[str, Any]:
    all_lat = []
    all_lon = []
    raw_dfs = {}

    for item in data_pairs:
        df_raw = parse_rtk_gga_raw(item["rtk"])
        raw_dfs[item["name"]] = df_raw

        all_lat.extend(df_raw["纬度_deg"].tolist())
        all_lon.extend(df_raw["经度_deg"].tolist())

    if len(all_lat) == 0:
        raise ValueError("全部 RTK 文件都没有有效经纬度，无法构建统一停车场场景坐标系。")

    lat0 = float(np.median(np.asarray(all_lat, dtype=float)))
    lon0 = float(np.median(np.asarray(all_lon, dtype=float)))

    all_x_enu = []
    all_y_enu = []

    for _, df_raw in raw_dfs.items():
        lat_arr = df_raw["纬度_deg"].to_numpy(dtype=float)
        lon_arr = df_raw["经度_deg"].to_numpy(dtype=float)
        xe = np.empty_like(lat_arr)
        yn = np.empty_like(lat_arr)
        for i in range(len(lat_arr)):
            xe[i], yn[i] = geodetic_to_local_xy(lat_arr[i], lon_arr[i], lat0, lon0)
        all_x_enu.append(xe)
        all_y_enu.append(yn)

    x_all = np.concatenate(all_x_enu)
    y_all = np.concatenate(all_y_enu)

    theta_scene = estimate_scene_rotation_from_points(x_all, y_all)

    x_scene_all, y_scene_all = rotate_xy(x_all, y_all, theta_scene)

    # 平移到正坐标，便于后续停车场区域划分
    scene_min_x = np.nanmin(x_scene_all)
    scene_min_y = np.nanmin(y_scene_all)
    tx = -scene_min_x
    ty = -scene_min_y

    return {
        "lat0": lat0,
        "lon0": lon0,
        "theta_scene_rad": theta_scene,
        "theta_scene_deg": math.degrees(theta_scene),
        "shift_x": tx,
        "shift_y": ty,
        "raw_dfs": raw_dfs,
    }

def read_rtk_series_to_scene(rtk_log_path: Path, scene_ref: Dict[str, Any]) -> pd.DataFrame:
    df_raw = parse_rtk_gga_raw(rtk_log_path)

    lat0 = scene_ref["lat0"]
    lon0 = scene_ref["lon0"]
    theta_scene = scene_ref["theta_scene_rad"]
    shift_x = scene_ref["shift_x"]
    shift_y = scene_ref["shift_y"]

    lat_arr = df_raw["纬度_deg"].to_numpy(dtype=float)
    lon_arr = df_raw["经度_deg"].to_numpy(dtype=float)

    E = np.empty_like(lat_arr)
    N = np.empty_like(lat_arr)
    for i in range(len(lat_arr)):
        E[i], N[i] = geodetic_to_local_xy(lat_arr[i], lon_arr[i], lat0, lon0)

    X_scene, Y_scene = rotate_xy(E, N, theta_scene)
    X_scene = X_scene + shift_x
    Y_scene = Y_scene + shift_y

    return pd.DataFrame({
        "时间_s": df_raw["时间_s"].to_numpy(dtype=float),
        "RTK_ENU_X_m": E,
        "RTK_ENU_Y_m": N,
        "RTK场景X_m": X_scene,
        "RTK场景Y_m": Y_scene,
        "纬度_deg": lat_arr,
        "经度_deg": lon_arr,
    })

# =========================================================
# 5. Geometry helpers
# =========================================================
def trilaterate_2d_three_anchors(anchors_xy: np.ndarray, d: np.ndarray) -> Tuple[float, float]:
    anchors_xy = np.asarray(anchors_xy, dtype=float)
    d = np.asarray(d, dtype=float)

    (x1, y1), (x2, y2), (x3, y3) = anchors_xy
    d1, d2, d3 = d

    A = np.array([
        [2 * (x2 - x1), 2 * (y2 - y1)],
        [2 * (x3 - x1), 2 * (y3 - y1)],
    ], dtype=float)

    b = np.array([
        (x2**2 - x1**2) + (y2**2 - y1**2) + (d1**2 - d2**2),
        (x3**2 - x1**2) + (y3**2 - y1**2) + (d1**2 - d3**2),
    ], dtype=float)

    if abs(np.linalg.det(A)) < 1e-12:
        return np.nan, np.nan

    try:
        sol = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return np.nan, np.nan
    return float(sol[0]), float(sol[1])

def circle_circle_intersections(
    c0: Tuple[float, float],
    r0: float,
    c1: Tuple[float, float],
    r1: float
) -> List[Tuple[float, float]]:
    x0, y0 = c0
    x1, y1 = c1

    dx = x1 - x0
    dy = y1 - y0
    d = math.hypot(dx, dy)

    if not np.isfinite(d) or d < 1e-12:
        return []
    if d > r0 + r1:
        return []
    if d < abs(r0 - r1):
        return []
    if d == 0 and abs(r0 - r1) < 1e-12:
        return []

    a = (r0**2 - r1**2 + d**2) / (2 * d)
    h2 = r0**2 - a**2
    if h2 < -1e-10:
        return []
    h = math.sqrt(max(h2, 0.0))

    xm = x0 + a * dx / d
    ym = y0 + a * dy / d

    rx = -dy * (h / d)
    ry = dx * (h / d)

    p1 = (xm + rx, ym + ry)
    p2 = (xm - rx, ym - ry)

    if math.hypot(p1[0] - p2[0], p1[1] - p2[1]) < 1e-10:
        return [p1]
    return [p1, p2]

def project_point_to_circle(
    center: Tuple[float, float],
    radius: float,
    point: Tuple[float, float]
) -> Tuple[float, float]:
    cx, cy = center
    px, py = point

    vx = px - cx
    vy = py - cy
    norm = math.hypot(vx, vy)

    if norm < 1e-12:
        return cx + radius, cy
    scale = radius / norm
    return cx + vx * scale, cy + vy * scale

def compute_cross_residual_single(x: float, y: float, d: np.ndarray, anchors_xy: np.ndarray) -> float:
    d = np.asarray(d, dtype=float)
    anchors_xy = np.asarray(anchors_xy, dtype=float)

    if not (np.isfinite(x) and np.isfinite(y) and np.all(np.isfinite(d))):
        return np.nan

    errs = []
    for i in range(3):
        ax, ay = anchors_xy[i]
        pred = math.sqrt((x - ax) ** 2 + (y - ay) ** 2)
        errs.append(pred - d[i])

    errs = np.asarray(errs, dtype=float)
    return float(np.sqrt(np.mean(errs ** 2)))

# =========================================================
# 6. Weak-prior stage 1 trajectory reconstruction
# =========================================================
def build_prediction(
    t: np.ndarray,
    x_prev: Optional[float],
    y_prev: Optional[float],
    x_prev2: Optional[float],
    y_prev2: Optional[float],
    i: int,
) -> Tuple[float, float]:
    if not is_valid_xy(x_prev, y_prev):
        return np.nan, np.nan

    if not is_valid_xy(x_prev2, y_prev2) or i < 2:
        return float(x_prev), float(y_prev)

    dt1 = max(t[i - 1] - t[i - 2], MIN_DT_FOR_VEL)
    dt2 = max(t[i] - t[i - 1], MIN_DT_FOR_VEL)

    vx = (float(x_prev) - float(x_prev2)) / dt1
    vy = (float(y_prev) - float(y_prev2)) / dt1

    x_pred = float(x_prev) + PREDICTION_BLEND * vx * dt2
    y_pred = float(y_prev) + PREDICTION_BLEND * vy * dt2
    return x_pred, y_pred

def limit_jump(
    x_candidate: float,
    y_candidate: float,
    x_prev: Optional[float],
    y_prev: Optional[float],
    dt: float,
    max_speed_mps: float = MAX_SPEED_MPS
) -> Tuple[float, float]:
    if not is_valid_xy(x_candidate, y_candidate):
        return np.nan, np.nan

    if not is_valid_xy(x_prev, y_prev):
        return float(x_candidate), float(y_candidate)

    max_dist = max_speed_mps * max(dt, MIN_DT_FOR_VEL)
    dx = float(x_candidate) - float(x_prev)
    dy = float(y_candidate) - float(y_prev)
    dist = math.hypot(dx, dy)

    if dist <= max_dist or dist < 1e-12:
        return float(x_candidate), float(y_candidate)

    scale = max_dist / dist
    return float(x_prev) + dx * scale, float(y_prev) + dy * scale

def solve_stage1_with_weak_prior(
    t: np.ndarray,
    df_ranges: pd.DataFrame,
    anchors_xy: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    vals = df_ranges[["range0(m)", "range1(m)", "range2(m)"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    n = len(vals)

    x = np.full(n, np.nan, dtype=float)
    y = np.full(n, np.nan, dtype=float)
    cross_residual = np.full(n, np.nan, dtype=float)
    source = np.full(n, "", dtype=object)

    prev_valid_indices: List[int] = []

    for i in range(n):
        row = vals[i]
        valid_idx = np.where(np.isfinite(row))[0]
        n_valid = len(valid_idx)

        x_prev = y_prev = x_prev2 = y_prev2 = None
        if len(prev_valid_indices) >= 1:
            j1 = prev_valid_indices[-1]
            x_prev, y_prev = float(x[j1]), float(y[j1])
        if len(prev_valid_indices) >= 2:
            j2 = prev_valid_indices[-2]
            x_prev2, y_prev2 = float(x[j2]), float(y[j2])

        x_pred, y_pred = build_prediction(t, x_prev, y_prev, x_prev2, y_prev2, i)

        if n_valid == 3:
            xi, yi = trilaterate_2d_three_anchors(anchors_xy, row)
            if is_valid_xy(xi, yi):
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                cross_residual[i] = compute_cross_residual_single(xi, yi, row, anchors_xy)
                source[i] = "direct_3range"
                prev_valid_indices.append(i)
            else:
                source[i] = "bad_3range"
            continue

        if n_valid == 2:
            a0, a1 = valid_idx.tolist()
            p0 = tuple(anchors_xy[a0])
            p1 = tuple(anchors_xy[a1])
            r0 = float(row[a0])
            r1 = float(row[a1])

            cands = circle_circle_intersections(p0, r0, p1, r1)
            if len(cands) > 0:
                if is_valid_xy(x_pred, y_pred):
                    dists = [math.hypot(cx - x_pred, cy - y_pred) for cx, cy in cands]
                    best_idx = int(np.argmin(dists))
                elif is_valid_xy(x_prev, y_prev):
                    dists = [math.hypot(cx - x_prev, cy - y_prev) for cx, cy in cands]
                    best_idx = int(np.argmin(dists))
                else:
                    best_idx = 0

                xi, yi = cands[best_idx]
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                source[i] = "prior_2range"
                prev_valid_indices.append(i)
            else:
                source[i] = "bad_2range"
            continue

        if n_valid == 1:
            a0 = int(valid_idx[0])
            p0 = tuple(anchors_xy[a0])
            r0 = float(row[a0])

            if is_valid_xy(x_pred, y_pred):
                xi, yi = project_point_to_circle(p0, r0, (x_pred, y_pred))
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                source[i] = "prior_1range"
                prev_valid_indices.append(i)
            elif is_valid_xy(x_prev, y_prev):
                xi, yi = project_point_to_circle(p0, r0, (x_prev, y_prev))
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                source[i] = "prior_1range"
                prev_valid_indices.append(i)
            else:
                source[i] = "none_1range_no_prior"
            continue

        source[i] = "none_0range"

    return x, y, cross_residual, source

# =========================================================
# 7. Fill short gaps for 0-range cases
# =========================================================
def fill_short_gaps_linear(
    t: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    source: np.ndarray,
    max_gap_points: int = 8
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    x2 = np.asarray(x, dtype=float).copy()
    y2 = np.asarray(y, dtype=float).copy()
    source2 = source.copy()

    n = len(x2)
    i = 0
    while i < n:
        if np.isfinite(x2[i]) and np.isfinite(y2[i]):
            i += 1
            continue

        j = i
        while j < n and not (np.isfinite(x2[j]) and np.isfinite(y2[j])):
            j += 1

        gap_len = j - i
        left = i - 1
        right = j

        can_fill = (
            gap_len <= max_gap_points and
            left >= 0 and right < n and
            np.isfinite(x2[left]) and np.isfinite(y2[left]) and
            np.isfinite(x2[right]) and np.isfinite(y2[right])
        )

        if can_fill:
            for k in range(i, j):
                alpha = (t[k] - t[left]) / max(t[right] - t[left], MIN_DT_FOR_VEL)
                x2[k] = (1 - alpha) * x2[left] + alpha * x2[right]
                y2[k] = (1 - alpha) * y2[left] + alpha * y2[right]
                if source2[k] in ["none_0range", "none_1range_no_prior", "bad_2range", "bad_3range"]:
                    source2[k] = "interp_gap"
        i = j

    return x2, y2, source2

# =========================================================
# 8. Stage 2 smoothing
# =========================================================
def smooth_trajectory_stage2(
    x: np.ndarray,
    y: np.ndarray,
    source: np.ndarray,
    window: int = 5
) -> Tuple[np.ndarray, np.ndarray]:
    if not SMOOTH_ENABLE:
        return np.asarray(x, dtype=float).copy(), np.asarray(y, dtype=float).copy()

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_s, y_s = smooth_xy(x, y, window=window)

    strong_mask = np.isin(source, ["direct_3range", "prior_2range"])
    x_s[strong_mask] = x[strong_mask]
    y_s[strong_mask] = y[strong_mask]

    return x_s, y_s

# =========================================================
# 9. Alignment
# =========================================================
def simple_flip_uwb(x: np.ndarray, y: np.ndarray, flip_x=True):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if flip_x:
        return -x, y
    return x.copy(), y.copy()

def kabsch_2d(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)

    Pc = P - P.mean(axis=0, keepdims=True)
    Qc = Q - Q.mean(axis=0, keepdims=True)

    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = Q.mean(axis=0) - (R @ P.mean(axis=0))
    theta_deg = math.degrees(math.atan2(R[1, 0], R[0, 0]))
    return R, t, theta_deg

def apply_rigid_transform(x, y, R, t):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    XY = np.column_stack([x, y]).astype(float)
    XY2 = (R @ XY.T).T + t
    return XY2[:, 0], XY2[:, 1]

def build_alignment_mask_strict(uwb_x, uwb_y, cross_residual, source, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    strong_source_mask = np.isin(source, ["direct_3range", "prior_2range"])
    residual_ok = np.isfinite(cross_residual) & (cross_residual <= MAX_CROSS_RESIDUAL_FOR_ALIGNMENT)
    prior2_ok = (source == "prior_2range")

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi) &
        strong_source_mask &
        (residual_ok | prior2_ok)
    )

def build_alignment_mask_relaxed(uwb_x, uwb_y, cross_residual, source, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    source_ok = np.isin(source, ["direct_3range", "prior_2range", "prior_1range", "interp_gap"])
    residual_ok = (~np.isfinite(cross_residual)) | (cross_residual <= MAX_CROSS_RESIDUAL_FOR_ALIGNMENT_RELAXED)

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi) &
        source_ok &
        residual_ok
    )

def build_alignment_mask_fallback_allfinite(uwb_x, uwb_y, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi)
    )

def evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, mask):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    if np.count_nonzero(mask) < 3:
        return None

    P = np.column_stack([uwb_x[mask], uwb_y[mask]])
    Q = np.column_stack([rtk_xi[mask], rtk_yi[mask]])

    R, t, theta_deg = kabsch_2d(P, Q)
    uwb_x2, uwb_y2 = apply_rigid_transform(uwb_x, uwb_y, R, t)
    score = calc_rmse(uwb_x2[mask], uwb_y2[mask], rtk_xi[mask], rtk_yi[mask])

    return {
        "R": R,
        "t": t,
        "theta_deg": float(theta_deg),
        "rmse": float(score),
        "uwb_x_aligned": uwb_x2,
        "uwb_y_aligned": uwb_y2,
    }

def search_best_time_shift_for_alignment(
    uwb_t, uwb_x, uwb_y, cross_residual, source, rtk_t, rtk_x, rtk_y
):
    uwb_t = np.asarray(uwb_t, dtype=float)
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_t = np.asarray(rtk_t, dtype=float)
    rtk_x = np.asarray(rtk_x, dtype=float)
    rtk_y = np.asarray(rtk_y, dtype=float)

    best = None
    best_score = np.inf

    def _search(time_shift_candidates):
        nonlocal best, best_score

        for dt in time_shift_candidates:
            rtk_xi, rtk_yi = interp2_at_times(rtk_t + dt, rtk_x, rtk_y, uwb_t)

            candidate = None
            mode = None
            align_mask = None

            m_strict = build_alignment_mask_strict(
                uwb_x=uwb_x,
                uwb_y=uwb_y,
                cross_residual=cross_residual,
                source=source,
                rtk_xi=rtk_xi,
                rtk_yi=rtk_yi,
            )
            if np.count_nonzero(m_strict) >= MIN_ALIGNMENT_POINTS:
                out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_strict)
                if out is not None:
                    candidate = out
                    mode = "strict"
                    align_mask = m_strict

            if candidate is None:
                m_relaxed = build_alignment_mask_relaxed(
                    uwb_x=uwb_x,
                    uwb_y=uwb_y,
                    cross_residual=cross_residual,
                    source=source,
                    rtk_xi=rtk_xi,
                    rtk_yi=rtk_yi,
                )
                if np.count_nonzero(m_relaxed) >= MIN_ALIGNMENT_POINTS_RELAXED:
                    out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_relaxed)
                    if out is not None:
                        candidate = out
                        mode = "relaxed"
                        align_mask = m_relaxed

            if candidate is None:
                m_all = build_alignment_mask_fallback_allfinite(uwb_x, uwb_y, rtk_xi, rtk_yi)
                if np.count_nonzero(m_all) >= 3:
                    out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_all)
                    if out is not None:
                        candidate = out
                        mode = "fallback_allfinite"
                        align_mask = m_all

            if candidate is None:
                continue

            score = candidate["rmse"]
            if score < best_score:
                best_score = score
                best = {
                    "time_shift_s": float(dt),
                    "R": candidate["R"],
                    "t": candidate["t"],
                    "theta_deg": candidate["theta_deg"],
                    "rmse": candidate["rmse"],
                    "rtk_x_interp": rtk_xi,
                    "rtk_y_interp": rtk_yi,
                    "uwb_x_aligned": candidate["uwb_x_aligned"],
                    "uwb_y_aligned": candidate["uwb_y_aligned"],
                    "align_mask": align_mask,
                    "align_mode": mode,
                    "align_points": int(np.count_nonzero(align_mask)),
                }

    _search(TIME_SHIFT_CANDIDATES_COARSE)
    if best is None:
        return None

    c = best["time_shift_s"]
    fine_candidates = np.arange(
        c - TIME_SHIFT_FINE_HALF_WIDTH,
        c + TIME_SHIFT_FINE_HALF_WIDTH + TIME_SHIFT_FINE_STEP,
        TIME_SHIFT_FINE_STEP
    )
    _search(fine_candidates)
    return best

# =========================================================
# 10. Plot
# =========================================================
def plot_aligned_trajectory(rtk_x, rtk_y, uwb_x_aligned, uwb_y_aligned, output_path: Path):
    ensure_parent_dir(output_path)

    rtk_x = pd.to_numeric(pd.Series(rtk_x), errors="coerce")
    rtk_y = pd.to_numeric(pd.Series(rtk_y), errors="coerce")
    uwb_x_aligned = pd.to_numeric(pd.Series(uwb_x_aligned), errors="coerce")
    uwb_y_aligned = pd.to_numeric(pd.Series(uwb_y_aligned), errors="coerce")

    plt.figure(figsize=(8, 8))

    m_rtk = rtk_x.notna() & rtk_y.notna()
    if m_rtk.any():
        plt.plot(rtk_x[m_rtk].to_numpy(), rtk_y[m_rtk].to_numpy(), label="RTK Scene Trajectory", linewidth=2.0)

    m_uwb = uwb_x_aligned.notna() & uwb_y_aligned.notna()
    if m_uwb.any():
        plt.plot(
            uwb_x_aligned[m_uwb].to_numpy(),
            uwb_y_aligned[m_uwb].to_numpy(),
            label="Aligned UWB Scene Trajectory",
            linewidth=1.5
        )

    plt.xlabel("Scene X (m)")
    plt.ylabel("Scene Y (m)")
    plt.title("Aligned UWB vs RTK in Parking-Lot Scene Coordinates")
    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=FIG_DPI)
    plt.close()

# =========================================================
# 10.1 Split the D4 output into D4 / D5 result tables
# =========================================================
def split_d4_d5_rows(df_out: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """按原有行顺序从中间拆分：奇数行时，D4 比 D5 多一行。"""
    middle = (len(df_out) + 1) // 2
    return df_out.iloc[:middle].copy(), df_out.iloc[middle:].copy()


def export_result_excel(
    name: str,
    df_out: pd.DataFrame,
    summary_df: pd.DataFrame,
    df_rtk: pd.DataFrame,
    output_xlsx: Path,
    log_path: Path,
) -> None:
    """D1~D3 沿用原输出；D4 同时生成 D4/D5 工作表与独立 Excel 文件。"""
    ensure_parent_dir(output_xlsx)
    if name != "4凹凸":
        with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
            df_out.to_excel(writer, sheet_name="逐点结果", index=False)
            summary_df.to_excel(writer, sheet_name="统计汇总", index=False)
            df_rtk.to_excel(writer, sheet_name="RTK原始转换结果", index=False)
        return

    df_d4, df_d5 = split_d4_d5_rows(df_out)

    # 原来的 D4 输出文件保留；将其中“逐点结果”替换为 D4、D5 两个工作表。
    # 配准变换、原始点序号和所有数据列完全沿用整段 D4 计算结果。
    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        df_d4.to_excel(writer, sheet_name="D4", index=False)
        df_d5.to_excel(writer, sheet_name="D5", index=False)
        summary_df.to_excel(writer, sheet_name="统计汇总", index=False)
        df_rtk.to_excel(writer, sheet_name="RTK原始转换结果", index=False)

    # 为后续分析方便，另外导出两个独立工作簿（不改变 D4 的整段配准）。
    d4_xlsx = OUTPUT_DIR / "D4_uwb_rtk_aligned_result.xlsx"
    d5_xlsx = OUTPUT_DIR / "D5_uwb_rtk_aligned_result.xlsx"
    df_d4.to_excel(d4_xlsx, sheet_name="D4", index=False, engine="openpyxl")
    df_d5.to_excel(d5_xlsx, sheet_name="D5", index=False, engine="openpyxl")

    log_print(
        log_path,
        f"[SPLIT] D4 在第 {len(df_d4)} 行之后拆分："
        f"D4={len(df_d4)} 行，D5={len(df_d5)} 行，总计={len(df_out)} 行。"
    )
    log_print(log_path, f"[SPLIT] D4 独立文件: {d4_xlsx}")
    log_print(log_path, f"[SPLIT] D5 独立文件: {d5_xlsx}")

# =========================================================
# 11. Single-file processing
# =========================================================
def process_one_file(
    name: str,
    uwb_input_path: Path,
    rtk_txt_path: Path,
    log_path: Path,
    scene_ref: Dict[str, Any],
) -> Dict[str, Any]:
    file_stem = uwb_input_path.stem
    output_xlsx = OUTPUT_DIR / f"{file_stem}_uwb_rtk_aligned_result.xlsx"
    output_plot = OUTPUT_DIR / f"{file_stem}_uwb_rtk_aligned_trajectory.png"

    if log_path.exists():
        log_path.unlink()

    log_print(log_path, "=" * 90)
    log_print(log_path, f"开始处理: {name}")
    log_print(log_path, f"UWB: {uwb_input_path}")
    log_print(log_path, f"RTK: {rtk_txt_path}")
    log_print(log_path, f"输出 Excel: {output_xlsx}")
    log_print(log_path, f"输出轨迹图: {output_plot}")

    if not uwb_input_path.exists():
        raise FileNotFoundError(f"UWB 文件不存在: {uwb_input_path}")
    if not rtk_txt_path.exists():
        raise FileNotFoundError(f"RTK 文件不存在: {rtk_txt_path}")

    uwb_t, df_uwb = robust_load_uwb_csv_3ranges(uwb_input_path)
    trimmed_rows = df_uwb.attrs.get("csv_corrected_trailing_empty_rows", 0)
    trimmed_header = df_uwb.attrs.get("csv_removed_header_empty_fields", 0)
    if trimmed_rows or trimmed_header:
        log_print(
            log_path,
            f"[UWB CSV] 已安全忽略表头末尾 {trimmed_header} 个空字段、"
            f"{trimmed_rows} 行末尾多余的空字段；原始时间与测距值保持不变。"
        )
    df_ranges = df_uwb[["range0(m)", "range1(m)", "range2(m)"]].copy()

    log_print(log_path, f"[UWB] Samples = {len(df_ranges)}, estimated rate = {infer_rate(uwb_t):.3f} Hz")
    log_print(log_path, "[DEBUG] UWB dtypes:")
    append_log(log_path, str(df_uwb.dtypes))
    log_print(log_path, "[DEBUG] UWB head:")
    append_log(log_path, str(df_uwb.head(3)))

    anchors_xy = np.array([
        ANCHORS_2D[0],
        ANCHORS_2D[1],
        ANCHORS_2D[2],
    ], dtype=float)

    uwb_x_stage1, uwb_y_stage1, cross_residual, uwb_source_stage1 = solve_stage1_with_weak_prior(
        t=uwb_t,
        df_ranges=df_ranges,
        anchors_xy=anchors_xy,
    )

    uwb_x_filled, uwb_y_filled, uwb_source_filled = fill_short_gaps_linear(
        t=uwb_t,
        x=uwb_x_stage1,
        y=uwb_y_stage1,
        source=uwb_source_stage1,
        max_gap_points=MAX_GAP_FILL_POINTS,
    )

    uwb_x_smooth, uwb_y_smooth = smooth_trajectory_stage2(
        x=uwb_x_filled,
        y=uwb_y_filled,
        source=uwb_source_filled,
        window=SMOOTH_WIN,
    )

    uwb_x_flip, uwb_y_flip = simple_flip_uwb(uwb_x_smooth, uwb_y_smooth, flip_x=UWB_FLIP_X)

    # 统一停车场场景坐标系下的 RTK
    df_rtk = read_rtk_series_to_scene(rtk_txt_path, scene_ref=scene_ref)
    rtk_t = df_rtk["时间_s"].to_numpy(dtype=float)
    rtk_x_scene = df_rtk["RTK场景X_m"].to_numpy(dtype=float)
    rtk_y_scene = df_rtk["RTK场景Y_m"].to_numpy(dtype=float)

    log_print(log_path, f"[RTK] Samples = {len(df_rtk)}, estimated rate = {infer_rate(rtk_t):.3f} Hz")
    log_print(log_path, "[DEBUG] RTK dtypes:")
    append_log(log_path, str(df_rtk.dtypes))
    log_print(log_path, "[DEBUG] RTK head:")
    append_log(log_path, str(df_rtk.head(3)))

    # UWB 直接配准到统一场景坐标系下的 RTK
    best = search_best_time_shift_for_alignment(
        uwb_t=uwb_t,
        uwb_x=uwb_x_flip,
        uwb_y=uwb_y_flip,
        cross_residual=cross_residual,
        source=uwb_source_filled,
        rtk_t=rtk_t,
        rtk_x=rtk_x_scene,
        rtk_y=rtk_y_scene,
    )

    if best is None:
        raise ValueError(f"配准失败：{name} 没有找到可用时间偏移/空间配准结果。")

    log_print(
        log_path,
        f"[ALIGN] mode = {best['align_mode']}, points = {best['align_points']}, "
        f"time_shift = {best['time_shift_s']:.3f}, rmse = {best['rmse']:.4f}"
    )

    df_out = pd.DataFrame({
        "点序号": np.arange(1, len(uwb_t) + 1),
        "时间_s": pd.to_numeric(pd.Series(uwb_t), errors="coerce"),
        "rangetime(ms)": pd.to_numeric(df_uwb["rangetime(ms)"], errors="coerce") if "rangetime(ms)" in df_uwb.columns else np.nan,

        "range0(m)": pd.to_numeric(df_ranges["range0(m)"], errors="coerce"),
        "range1(m)": pd.to_numeric(df_ranges["range1(m)"], errors="coerce"),
        "range2(m)": pd.to_numeric(df_ranges["range2(m)"], errors="coerce"),

        "有效测距个数": np.isfinite(
            df_ranges[["range0(m)", "range1(m)", "range2(m)"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
        ).sum(axis=1),

        "第一阶段UWB_X_m": pd.to_numeric(pd.Series(uwb_x_stage1), errors="coerce"),
        "第一阶段UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_stage1), errors="coerce"),

        "补全后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_filled), errors="coerce"),
        "补全后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_filled), errors="coerce"),

        "平滑后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_smooth), errors="coerce"),
        "平滑后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_smooth), errors="coerce"),

        "翻转后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_flip), errors="coerce"),
        "翻转后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_flip), errors="coerce"),

        "UWB位置来源": pd.Series(uwb_source_filled).astype(str),
        "三圆交叉残差_m": pd.to_numeric(pd.Series(cross_residual), errors="coerce"),

        "RTK插值场景X_m": pd.to_numeric(pd.Series(best["rtk_x_interp"]), errors="coerce"),
        "RTK插值场景Y_m": pd.to_numeric(pd.Series(best["rtk_y_interp"]), errors="coerce"),

        "UWB配准后场景X_m": pd.to_numeric(pd.Series(best["uwb_x_aligned"]), errors="coerce"),
        "UWB配准后场景Y_m": pd.to_numeric(pd.Series(best["uwb_y_aligned"]), errors="coerce"),

        "是否参与配准": pd.Series(best["align_mask"]).astype(int),
    })

    df_out["定位误差X_m"] = to_num_series(df_out["UWB配准后场景X_m"]) - to_num_series(df_out["RTK插值场景X_m"])
    df_out["定位误差Y_m"] = to_num_series(df_out["UWB配准后场景Y_m"]) - to_num_series(df_out["RTK插值场景Y_m"])
    df_out["定位误差_RTK_m"] = np.sqrt(df_out["定位误差X_m"] ** 2 + df_out["定位误差Y_m"] ** 2)

    err_s = to_num_series(df_out["定位误差_RTK_m"])
    res_s = to_num_series(df_out["三圆交叉残差_m"])

    valid_err = err_s.notna()
    valid_res = res_s.notna()

    source_counts = df_out["UWB位置来源"].astype(str).value_counts(dropna=False).to_dict()

    # 原始 RTK 场景坐标也一起写出，便于你检查停车场构建结果
    summary_row = {
        "数据组": name,
        "状态": "成功",
        "错误信息": "",
        "日志文件": str(log_path),
        "UWB输入文件": str(uwb_input_path),
        "RTK文件": str(rtk_txt_path),
        "总样本数": len(df_out),
        "可得到UWB位置点数": finite_count_2cols(df_out["UWB配准后场景X_m"], df_out["UWB配准后场景Y_m"]),
        "有RTK对应点数": finite_count_2cols(df_out["RTK插值场景X_m"], df_out["RTK插值场景Y_m"]),
        "可计算误差点数": int(valid_err.sum()),
        "最佳时间偏移_s": float(best["time_shift_s"]),
        "旋转角_deg": float(best["theta_deg"]),
        "平移X_m": float(best["t"][0]),
        "平移Y_m": float(best["t"][1]),
        "配准RMSE_m": float(best["rmse"]),
        "配准模式": str(best["align_mode"]),
        "配准点数": int(best["align_points"]),
        "三圆交叉残差均值_m": float(res_s[valid_res].mean()) if valid_res.any() else np.nan,
        "三圆交叉残差RMSE_m": float(np.sqrt((res_s[valid_res] ** 2).mean())) if valid_res.any() else np.nan,
        "定位误差_RTK均值_m": float(err_s[valid_err].mean()) if valid_err.any() else np.nan,
        "定位误差_RTK_RMSE_m": float(np.sqrt((err_s[valid_err] ** 2).mean())) if valid_err.any() else np.nan,
        "定位误差_RTK_95分位_m": float(err_s[valid_err].quantile(0.95)) if valid_err.any() else np.nan,
        "direct_3range点数": int(source_counts.get("direct_3range", 0)),
        "prior_2range点数": int(source_counts.get("prior_2range", 0)),
        "prior_1range点数": int(source_counts.get("prior_1range", 0)),
        "interp_gap点数": int(source_counts.get("interp_gap", 0)),
        "none_0range点数": int(source_counts.get("none_0range", 0)),
        "输出Excel": str(output_xlsx),
        "输出轨迹图": str(output_plot),
    }

    summary_df = pd.DataFrame([summary_row])

    export_result_excel(
        name=name,
        df_out=df_out,
        summary_df=summary_df,
        df_rtk=df_rtk,
        output_xlsx=output_xlsx,
        log_path=log_path,
    )

    plot_aligned_trajectory(
        rtk_x=best["rtk_x_interp"],
        rtk_y=best["rtk_y_interp"],
        uwb_x_aligned=best["uwb_x_aligned"],
        uwb_y_aligned=best["uwb_y_aligned"],
        output_path=output_plot,
    )

    log_print(log_path, "Processing completed.")
    log_print(log_path, f"Best time shift (s): {best['time_shift_s']:.3f}")
    log_print(log_path, f"Alignment RMSE (m): {best['rmse']:.4f}")
    log_print(log_path, f"Excel saved to: {output_xlsx}")
    log_print(log_path, f"Plot saved to: {output_plot}")

    return summary_row

# =========================================================
# 12. Main
# =========================================================
def main():
    ensure_dir(OUTPUT_DIR)
    print(f"Output dir: {OUTPUT_DIR}")
    print(f"Batch summary: {OUTPUT_SUMMARY_XLSX}")

    # 先构建统一停车场场景坐标系
    scene_ref = build_global_parking_scene(DATA_PAIRS)

    scene_info_df = pd.DataFrame([{
        "全局原点纬度_deg": scene_ref["lat0"],
        "全局原点经度_deg": scene_ref["lon0"],
        "场景旋转角_deg": scene_ref["theta_scene_deg"],
        "场景平移X_m": scene_ref["shift_x"],
        "场景平移Y_m": scene_ref["shift_y"],
        "说明": "所有RTK已统一到该停车场场景坐标系，可直接用于后续分区与5x5网格划分",
    }])
    scene_info_df.to_excel(OUTPUT_SCENE_INFO_XLSX, index=False)

    all_summary_rows = []

    for item in DATA_PAIRS:
        log_path = OUTPUT_DIR / f"{item['name']}_run_log.txt"

        try:
            result = process_one_file(
                name=item["name"],
                uwb_input_path=item["uwb"],
                rtk_txt_path=item["rtk"],
                log_path=log_path,
                scene_ref=scene_ref,
            )
            all_summary_rows.append(result)

        except Exception as e:
            err_msg = str(e)
            tb = traceback.format_exc()

            print("=" * 90)
            print(f"[ERROR] 文件处理失败: {item['name']}")
            print(f"UWB: {item['uwb']}")
            print(f"RTK: {item['rtk']}")
            print(f"原因: {err_msg}")
            print(tb)

            append_log(log_path, "=" * 90)
            append_log(log_path, f"[ERROR] 文件处理失败: {item['name']}")
            append_log(log_path, f"UWB: {item['uwb']}")
            append_log(log_path, f"RTK: {item['rtk']}")
            append_log(log_path, f"原因: {err_msg}")
            append_log(log_path, tb)

            all_summary_rows.append({
                "数据组": item["name"],
                "状态": "失败",
                "错误信息": err_msg,
                "日志文件": str(log_path),
                "UWB输入文件": str(item["uwb"]),
                "RTK文件": str(item["rtk"]),
                "总样本数": np.nan,
                "可得到UWB位置点数": np.nan,
                "有RTK对应点数": np.nan,
                "可计算误差点数": np.nan,
                "最佳时间偏移_s": np.nan,
                "旋转角_deg": np.nan,
                "平移X_m": np.nan,
                "平移Y_m": np.nan,
                "配准RMSE_m": np.nan,
                "配准模式": "",
                "配准点数": np.nan,
                "三圆交叉残差均值_m": np.nan,
                "三圆交叉残差RMSE_m": np.nan,
                "定位误差_RTK均值_m": np.nan,
                "定位误差_RTK_RMSE_m": np.nan,
                "定位误差_RTK_95分位_m": np.nan,
                "direct_3range点数": np.nan,
                "prior_2range点数": np.nan,
                "prior_1range点数": np.nan,
                "interp_gap点数": np.nan,
                "none_0range点数": np.nan,
                "输出Excel": "",
                "输出轨迹图": "",
            })

    df_batch_summary = pd.DataFrame(all_summary_rows)
    df_batch_summary.to_excel(OUTPUT_SUMMARY_XLSX, index=False)

    print("=" * 90)
    print("全部处理完成。")
    print(f"批量汇总已保存到: {OUTPUT_SUMMARY_XLSX}")
    print(f"停车场统一场景参考已保存到: {OUTPUT_SCENE_INFO_XLSX}")

if __name__ == "__main__":
    main()


Output dir: D:\Desktop\missing_output\batch_output
Batch summary: D:\Desktop\missing_output\batch_output\uwb_rtk_batch_summary.xlsx
开始处理: 1试验
UWB: D:\Desktop\missing_output\D1_missing.csv
RTK: D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\1.txt
输出 Excel: D:\Desktop\missing_output\batch_output\D1_missing_uwb_rtk_aligned_result.xlsx
输出轨迹图: D:\Desktop\missing_output\batch_output\D1_missing_uwb_rtk_aligned_trajectory.png
[UWB] Samples = 1435, estimated rate = 8.929 Hz
[DEBUG] UWB dtypes:
[DEBUG] UWB head:
[RTK] Samples = 2896, estimated rate = 10.000 Hz
[DEBUG] RTK dtypes:
[DEBUG] RTK head:
[ALIGN] mode = strict, points = 1275, time_shift = -10.300, rmse = 0.7666
Processing completed.
Best time shift (s): -10.300
Alignment RMSE (m): 0.7666
Excel saved to: D:\Desktop\missing_output\batch_output\D1_missing_uwb_rtk_aligned_result.xlsx
Plot saved to: D:\Desktop\missing_output\batch_output\D1_missing_uwb_rtk_aligned_trajectory.png
开始处理: 2圆形
UWB: D:\Desktop\missing_output\D2_

# 划分网格

In [20]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import shutil
import traceback
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# 1. 用户设置（保留 MATLAB 原程序的输入、输出和统计参数）
# =========================================================
BATCH_DIR = Path(r"D:\Desktop\missing_output\batch_output")
# 默认优先读取拆分后的两个独立文件；若缺失，自动改读原 D4 工作簿中 D4/D5 工作表。
D4_COMBINED_FILE = BATCH_DIR / "D4_missing_uwb_rtk_aligned_result.xlsx"
FILE_LIST = [
    BATCH_DIR / "D1_missing_uwb_rtk_aligned_result.xlsx",
    BATCH_DIR / "D2_missing_uwb_rtk_aligned_result.xlsx",
    BATCH_DIR / "D3_missing_uwb_rtk_aligned_result.xlsx",
    BATCH_DIR / "D4_uwb_rtk_aligned_result.xlsx",
    BATCH_DIR / "D5_uwb_rtk_aligned_result.xlsx",
]
SHEET_NAME = "逐点结果"
ROOT_OUT_DIR = Path(r"D:\Desktop\AllRegionAnalysisOutput")

CELL_SIZE = 5.0               # 5 m 网格
ERROR_THRESHOLD = 0.5          # 二维误差严格小于 0.5 m 才算达标
PATH_BREAK_THRESHOLD = 2.5    # 同一区域内相邻点间距超过该值即断开路径
WRITE_REGION_EXCEL = True     # 每个区域、每组数据输出一个 Excel
REGION_EXCEL_WARNING_COUNT = 350

# 绘图设置：与 MATLAB 原程序一致；只输出 PNG，不打开交互式窗口。
MAKE_TRAJECTORY_PLOT = True
SHOW_REGION_GRID = True
SHOW_REGION_LABELS = True
SHOW_ERROR_COUNT_ON_MAP = True
MAX_GRID_LINES = 180
MAX_REGION_LABELS = 120
MAX_DRAW_POINTS_PER_LINE = 100_000
PLOT_RESOLUTION = 150

RANGE_COLS = ["range0(m)", "range1(m)", "range2(m)"]
RANGETIME_COL = "rangetime(ms)"  # 可选列：仅原样保留，不参与计算或排序
RES_COL = "三圆交叉残差_m"
UWB_X_COL = "UWB配准后场景X_m"
UWB_Y_COL = "UWB配准后场景Y_m"
RTK_X_COL = "RTK插值场景X_m"
RTK_Y_COL = "RTK插值场景Y_m"
ERR_COL = "UWB_RTK_Error_m"
REQUIRED_COLS = RANGE_COLS + [RES_COL, UWB_X_COL, UWB_Y_COL, RTK_X_COL, RTK_Y_COL]
STAT_COLS = [
    "range0_有效率%", "range1_有效率%", "range2_有效率%",
    "最大间距率%", "三圆交叉残差均值_m", "路径条数", "总行数",
    "误差阈值_m", "误差小于阈值点数", "有效误差点数",
    "UWB_RTK_误差总和_m",
]


def log_step(log_path: Path, fmt: str, *args) -> None:
    """与 MATLAB 的 logStep 一样：命令行和无时间前缀的日志同步输出。"""
    message = fmt % args if args else fmt
    print(message, flush=True)
    with log_path.open("a", encoding="utf-8") as stream:
        stream.write(message + "\n")


def max_missing_ratio(values: np.ndarray) -> float:
    """最长连续缺失段长度 / 本路径的总点数；NaN、正负无穷算缺失。"""
    values = np.asarray(values, dtype=float).reshape(-1)
    if values.size == 0:
        return 0.0
    missing = ~np.isfinite(values)
    flags = np.diff(np.r_[False, missing, False].astype(np.int8))
    begins = np.flatnonzero(flags == 1)
    if begins.size == 0:
        return 0.0
    ends = np.flatnonzero(flags == -1)
    return float(np.max(ends - begins) / values.size)


def matlab_round(values: np.ndarray, digits: int) -> np.ndarray:
    """按 MATLAB round 习惯将恰好落在 0.5 的数向远离 0 的方向舍入。"""
    array = np.asarray(values, dtype=float)
    scale = 10.0 ** digits
    with np.errstate(invalid="ignore", over="ignore"):
        return np.copysign(np.floor(np.abs(array) * scale + 0.5), array) / scale


def get_default_file_list() -> list[Path]:
    """兼容独立 D4/D5 文件和原 D4 工作簿的双工作表形式。"""
    paths = FILE_LIST.copy()
    for idx in (3, 4):
        if not paths[idx].is_file() and D4_COMBINED_FILE.is_file():
            paths[idx] = D4_COMBINED_FILE
    return paths


def choose_data_sheet(excel: pd.ExcelFile, d_number: int, force_sheet: str | None = None) -> str:
    """自动挑选包含定位必需列的工作表，绝不误用统计汇总页。"""
    names = excel.sheet_names
    if force_sheet is not None:
        if force_sheet not in names:
            raise ValueError(
                f"D{d_number} 与另一组共用一个文件，但缺少「{force_sheet}」工作表。"
                f"当前工作表：{names}。请先确认 D4 已正确拆分成 D4、D5。"
            )
        candidates = [force_sheet]
    else:
        preferred = [f"D{d_number}", SHEET_NAME] if d_number in (4, 5) else [SHEET_NAME, f"D{d_number}"]
        candidates = [name for name in preferred if name in names]
        candidates += [name for name in names if name not in candidates]

    for name in candidates:
        # 只读取表头检查列名，避免把“统计汇总”误认为逐点数据。
        headers = pd.read_excel(excel, sheet_name=name, nrows=0).columns
        if set(REQUIRED_COLS).issubset(headers):
            return name

    raise ValueError(
        f"D{d_number} 文件中没有包含全部定位必需列的工作表。"
        f"现有工作表：{names}；必需列：{REQUIRED_COLS}。"
    )


def read_input_excel(
    path: Path,
    d_number: int,
    error_threshold: float,
    force_sheet: str | None = None,
) -> tuple[pd.DataFrame, np.ndarray]:
    """读取正确工作表，重算每行误差，返回逐点数据及有效 UWB 行号。"""
    if not path.is_file():
        raise FileNotFoundError(f"找不到输入文件：{path}")
    with pd.ExcelFile(path, engine="openpyxl") as excel:
        sheet = choose_data_sheet(excel, d_number, force_sheet)
        data = pd.read_excel(excel, sheet_name=sheet)
    missing = [c for c in REQUIRED_COLS if c not in data.columns]
    if missing:
        raise ValueError(f"D{d_number} 缺少必需的列：{'、'.join(missing)}")
    # 原始 rangetime(ms) 与输入行一一对应；每个 Region Excel 都原样保留该列。
    keep_cols = REQUIRED_COLS + ([RANGETIME_COL] if RANGETIME_COL in data.columns else [])
    data = data.loc[:, keep_cols].copy()
    for col in REQUIRED_COLS:
        original = data[col]
        # 允许 Excel 空单元格，禁止将非空且无法转成实数的内容静默丢掉。
        converted = pd.to_numeric(original, errors="coerce")
        bad = original.notna() & converted.isna()
        if bool(bad.any()):
            raise ValueError(f"D{d_number} 的 {col} 列包含非数值内容（示例：{original[bad].iloc[0]!r}）。")
        if np.iscomplexobj(converted.to_numpy()):
            raise ValueError(f"D{d_number} 的 {col} 列必须为实数型数值。")
        data[col] = converted.astype(float)

    ux = data[UWB_X_COL].to_numpy(dtype=float)
    uy = data[UWB_Y_COL].to_numpy(dtype=float)
    rx = data[RTK_X_COL].to_numpy(dtype=float)
    ry = data[RTK_Y_COL].to_numpy(dtype=float)
    valid_uwb = np.isfinite(ux) & np.isfinite(uy)
    valid_rtk = np.isfinite(rx) & np.isfinite(ry)
    error = np.hypot(ux - rx, uy - ry)
    error[~(valid_uwb & valid_rtk)] = np.nan
    data[ERR_COL] = error
    # 严格小于阈值；阈值处及舍入意义下极接近阈值处不计入。
    tolerance = 8.0 * np.spacing(max(1.0, abs(error_threshold)))
    data["ErrorBelowThreshold"] = np.isfinite(error) & (error < error_threshold - tolerance)
    data.attrs["source_sheet"] = sheet
    return data, np.flatnonzero(valid_uwb)


def safe_plot_xy(
    x: np.ndarray, y: np.ndarray, max_points: int
) -> tuple[np.ndarray, np.ndarray]:
    """轨迹抽稀，同时保留缺失点前后的断点，防止跨缺失段错误连线。"""
    x = np.asarray(x, dtype=float).reshape(-1).copy()
    y = np.asarray(y, dtype=float).reshape(-1).copy()
    if x.shape != y.shape:
        raise ValueError("轨迹 X/Y 的长度必须相同。")
    if max_points < 1:
        raise ValueError("max_points 必须大于 0。")
    valid = np.isfinite(x) & np.isfinite(y)
    x[~valid] = np.nan
    y[~valid] = np.nan
    n = x.size
    if n <= max_points:
        return x, y
    step = int(np.ceil(n / max_points))
    # 这些索引保住 NaN 两侧的有效点，以及缺失段边界的 NaN。
    boundaries = np.flatnonzero(valid[1:] != valid[:-1])
    selected = np.unique(np.r_[np.arange(0, n, step), boundaries, boundaries + 1, n - 1])
    return x[selected], y[selected]


def find_cjk_plot_font() -> str | None:
    """优先查找 Windows/中文环境中的中文字库；没有时使用英文图注。"""
    from matplotlib import font_manager

    names = {font.name for font in font_manager.fontManager.ttflist}
    for candidate in (
        "Microsoft YaHei", "SimHei", "DengXian", "PingFang SC",
        "Noto Sans CJK SC", "Source Han Sans SC", "WenQuanYi Micro Hei",
        "Noto Sans CJK JP", "Arial Unicode MS",
    ):
        if candidate in names:
            return candidate
    return None


def draw_trajectory_map(
    all_tables: list[pd.DataFrame],
    region_rc: np.ndarray,
    pooled_total: np.ndarray,
    pooled_below: np.ndarray,
    xmin: float, xmax: float, ymin: float, ymax: float,
    n_cols: int, n_rows: int,
    cell_size: float, error_threshold: float,
    root_out_dir: Path, log_path: Path,
    show_region_grid: bool = SHOW_REGION_GRID,
    show_region_labels: bool = SHOW_REGION_LABELS,
    show_error_count_on_map: bool = SHOW_ERROR_COUNT_ON_MAP,
    max_grid_lines: int = MAX_GRID_LINES,
    max_region_labels: int = MAX_REGION_LABELS,
    max_draw_points_per_line: int = MAX_DRAW_POINTS_PER_LINE,
    plot_resolution: int = PLOT_RESOLUTION,
) -> Path:
    """根据 MATLAB 绘图逻辑，导出 UWB 实线/RTK 虚线及区域网格 PNG。"""
    if max_grid_lines < 0 or max_region_labels < 0:
        raise ValueError("网格线和区域标签数量上限不能为负数。")
    if max_draw_points_per_line < 1 or plot_resolution < 1:
        raise ValueError("抽稀点数与绘图分辨率必须是正整数。")

    # Figure + Agg Canvas 不依赖 GUI，适合在服务器、终端或 Windows 运行。
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    from matplotlib import colormaps

    font_name = find_cjk_plot_font()
    if font_name:
        from matplotlib.font_manager import FontProperties
        cn_font = FontProperties(family=font_name)
        def label(text_cn: str, text_en: str) -> str:
            return text_cn
    else:
        cn_font = None
        def label(text_cn: str, text_en: str) -> str:
            return text_en
        log_step(log_path, "绘图：未检测到中文字体，坐标轴及区域文字改用英文以避免乱码。")

    fig = Figure(figsize=(10.8, 7.4), facecolor="white")
    FigureCanvasAgg(fig)
    ax = fig.add_subplot(111)
    fig.subplots_adjust(left=0.11, right=0.78, bottom=0.12, top=0.88)

    if show_region_grid:
        if n_rows + n_cols + 2 <= max_grid_lines:
            for j in range(n_cols + 1):
                xx = xmin + j * cell_size
                ax.plot([xx, xx], [ymin, ymax], color=(0.85, 0.85, 0.85),
                        linewidth=0.6, zorder=0, label="_nolegend_")
            for i in range(n_rows + 1):
                yy = ymin + i * cell_size
                ax.plot([xmin, xmax], [yy, yy], color=(0.85, 0.85, 0.85),
                        linewidth=0.6, zorder=0, label="_nolegend_")
        else:
            log_step(log_path, "绘图：网格线共 %d 条，超过上限 %d，已跳过网格。",
                     n_rows + n_cols + 2, max_grid_lines)

    palette = colormaps.get_cmap("tab10")
    for f, table in enumerate(all_tables, 1):
        color = palette((f - 1) % 10)
        ux, uy = safe_plot_xy(table[UWB_X_COL].to_numpy(),
                              table[UWB_Y_COL].to_numpy(), max_draw_points_per_line)
        rx, ry = safe_plot_xy(table[RTK_X_COL].to_numpy(),
                              table[RTK_Y_COL].to_numpy(), max_draw_points_per_line)
        if np.isfinite(ux).any():
            ax.plot(ux, uy, "-", color=color, linewidth=1.8,
                    zorder=2, label=f"D{f} UWB")
        if np.isfinite(rx).any():
            ax.plot(rx, ry, "--", color=color, linewidth=1.5,
                    zorder=2, label=f"D{f} RTK")

    n_regions = len(region_rc)
    if show_region_labels:
        if n_regions <= max_region_labels:
            for k, (grid_row, grid_col) in enumerate(region_rc, 1):
                cx = xmin + (grid_col - 0.5) * cell_size
                cy = ymin + (grid_row - 0.5) * cell_size
                text_opts = {"ha": "center", "va": "center", "zorder": 4}
                if cn_font:
                    text_opts["fontproperties"] = cn_font
                ax.text(cx, cy + 0.55, f"#{k}", fontsize=9,
                        fontweight="bold", color="0.2", **text_opts)
                if show_error_count_on_map:
                    t = label(f"总{int(pooled_total[k-1])} / 达标{int(pooled_below[k-1])}",
                              f"All {int(pooled_total[k-1])} / Pass {int(pooled_below[k-1])}")
                else:
                    t = label(f"总{int(pooled_total[k-1])}点",
                              f"All {int(pooled_total[k-1])}")
                ax.text(cx, cy - 0.55, t, fontsize=8, color="0.4", **text_opts)
        else:
            log_step(log_path, "绘图：区域数 %d 超过标签上限 %d，已跳过区域标签。",
                     n_regions, max_region_labels)

    font_args = {"fontproperties": cn_font} if cn_font else {}
    ax.set_xlabel(label("场景 X（米）", "Scene X (m)"), **font_args)
    ax.set_ylabel(label("场景 Y（米）", "Scene Y (m)"), **font_args)
    ax.set_title(
        label(f"UWB（实线）与 RTK（虚线）轨迹 | 网格 {cell_size:g} 米 | 达标阈值 {error_threshold:g} 米",
              f"UWB (solid) & RTK (dashed) | Grid {cell_size:g} m | Threshold {error_threshold:g} m"),
        fontsize=12, **font_args,
    )
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
                  frameon=False, fontsize=9)

    image_path = root_out_dir / "Region_Map_UWB_RTK_Trajectories.png"
    fig.savefig(image_path, dpi=plot_resolution, facecolor="white")
    fig.clear()  # 释放大轨迹占用的内存
    log_step(log_path, "绘图：已保存 PNG：%s", image_path)
    return image_path


def run_analysis(
    file_list=None,
    root_out_dir=None,
    cell_size: float = CELL_SIZE,
    error_threshold: float = ERROR_THRESHOLD,
    path_break_threshold: float = PATH_BREAK_THRESHOLD,
    write_region_excel: bool = WRITE_REGION_EXCEL,
    make_trajectory_plot: bool = MAKE_TRAJECTORY_PLOT,
    show_region_grid: bool = SHOW_REGION_GRID,
    show_region_labels: bool = SHOW_REGION_LABELS,
    show_error_count_on_map: bool = SHOW_ERROR_COUNT_ON_MAP,
    max_grid_lines: int = MAX_GRID_LINES,
    max_region_labels: int = MAX_REGION_LABELS,
    max_draw_points_per_line: int = MAX_DRAW_POINTS_PER_LINE,
    plot_resolution: int = PLOT_RESOLUTION,
) -> dict:
    """完成区域划分、Excel 输出和轨迹 PNG 绘图。"""
    file_list = [Path(p) for p in (get_default_file_list() if file_list is None else file_list)]
    root_out_dir = Path(ROOT_OUT_DIR if root_out_dir is None else root_out_dir)
    if not file_list:
        raise ValueError("至少需要一个输入 Excel 文件。")
    for label, value in (
        ("cell_size", cell_size),
        ("error_threshold", error_threshold),
        ("path_break_threshold", path_break_threshold),
    ):
        if not np.isfinite(value) or (value <= 0 if label == "cell_size" else value < 0):
            raise ValueError(f"{label} 必须是有限的有效非负数；cell_size 必须大于 0。")

    root_out_dir.mkdir(parents=True, exist_ok=True)
    region_out_dir = root_out_dir / "RegionOutput"
    stat_out_dir = root_out_dir / "ValidRate_Output"
    stat_out_dir.mkdir(parents=True, exist_ok=True)
    log_path = root_out_dir / "LastRun_Log.txt"
    log_path.write_text("", encoding="utf-8")
    log_step(log_path, "程序开始运行，固定输出目录：%s", root_out_dir)

    try:
        # 1. 读取 D1~D5，并从有效 UWB 点求所有文件的共同边界。
        n_files = len(file_list)
        all_tables: list[pd.DataFrame] = []
        valid_uwb_rows: list[np.ndarray] = []
        min_x = min_y = np.inf
        max_x = max_y = -np.inf
        log_step(log_path, "开始读取输入数据：共 %d 个 Excel 数据组。", n_files)
        # 当 D4/D5 指向同一个 Excel，强制分别读取 D4/D5，防止重复统计整份 D4。
        shared_d4_d5 = (
            n_files >= 5 and file_list[3].resolve() == file_list[4].resolve()
        )
        for f, file_path in enumerate(file_list, 1):
            force_sheet = f"D{f}" if shared_d4_d5 and f in (4, 5) else None
            table, rows = read_input_excel(file_path, f, error_threshold, force_sheet)
            log_step(log_path, "D%d：从 %s 读取工作表「%s」，共 %d 行。", f, file_path, table.attrs["source_sheet"], len(table))
            all_tables.append(table)
            valid_uwb_rows.append(rows)
            if rows.size:
                ux = table[UWB_X_COL].to_numpy(dtype=float)[rows]
                uy = table[UWB_Y_COL].to_numpy(dtype=float)[rows]
                min_x, max_x = min(min_x, float(ux.min())), max(max_x, float(ux.max()))
                min_y, max_y = min(min_y, float(uy.min())), max(max_y, float(uy.max()))

        if not any(len(rows) for rows in valid_uwb_rows):
            raise ValueError("所有输入文件均无有效的 UWB 配准坐标，无法划分网格。")

        # 2. 5 m 网格：只建立包含有效 UWB 点的区域，编号按行、列递增。
        xmin = np.floor(min_x / cell_size) * cell_size
        ymin = np.floor(min_y / cell_size) * cell_size
        xmax = np.ceil(max_x / cell_size) * cell_size
        ymax = np.ceil(max_y / cell_size) * cell_size
        if xmax <= xmin:
            xmax = xmin + cell_size
        if ymax <= ymin:
            ymax = ymin + cell_size
        n_cols = int(round((xmax - xmin) / cell_size))
        n_rows = int(round((ymax - ymin) / cell_size))
        log_step(log_path, "网格范围：X 从 %.3f 到 %.3f 米，Y 从 %.3f 到 %.3f 米。", xmin, xmax, ymin, ymax)

        coords_by_file = []
        for table, rows in zip(all_tables, valid_uwb_rows):
            if not rows.size:
                coords_by_file.append(np.empty((0, 2), dtype=np.int64))
                continue
            px = table[UWB_X_COL].to_numpy(dtype=float)[rows]
            py = table[UWB_Y_COL].to_numpy(dtype=float)[rows]
            cols = np.clip(np.floor((px - xmin) / cell_size).astype(np.int64) + 1, 1, n_cols)
            grid_rows = np.clip(np.floor((py - ymin) / cell_size).astype(np.int64) + 1, 1, n_rows)
            coords_by_file.append(np.column_stack((grid_rows, cols)))

        region_rc, point_region_zero_based = np.unique(
            np.concatenate(coords_by_file, axis=0), axis=0, return_inverse=True
        )
        n_regions = region_rc.shape[0]
        region_of_file_row = []
        offset = 0
        for table, rows in zip(all_tables, valid_uwb_rows):
            group = np.zeros(len(table), dtype=np.int64)
            nr = len(rows)
            group[rows] = point_region_zero_based[offset:offset + nr] + 1
            region_of_file_row.append(group)
            offset += nr
        log_step(log_path, "实际包含数据的区域：%d 个（候选区域共 %d 个）。", n_regions, n_rows * n_cols)

        # 3. 统计数组：维度为「实际区域 × D组 × 三个测距基站」。
        valid_rate = np.zeros((n_regions, n_files, 3), dtype=float)
        total_count = np.zeros((n_regions, n_files), dtype=np.int64)
        max_gap_ratio = np.zeros((n_regions, n_files, 3), dtype=float)
        obj_max_gap_ratio = np.zeros((n_regions, n_files), dtype=float)
        res_mean = np.full((n_regions, n_files), np.nan, dtype=float)
        path_count = np.zeros((n_regions, n_files), dtype=np.int64)
        err_below_count = np.zeros((n_regions, n_files), dtype=np.int64)
        err_valid_count = np.zeros((n_regions, n_files), dtype=np.int64)
        # 每个专家（D 组）在每个区域的二维定位误差累计值，单位 m
        err_sum = np.zeros((n_regions, n_files), dtype=float)

        n_potential_xlsx = sum(len(np.unique(g[g > 0])) for g in region_of_file_row)
        if write_region_excel and n_potential_xlsx > REGION_EXCEL_WARNING_COUNT:
            log_step(log_path, "提示：即将导出 %d 个区域 Excel 文件，可能耗时较长；文件仍将全部导出。", n_potential_xlsx)

        # 确认输入及区域编号后再清除旧 RegionOutput，避免异常读取时误删。
        if region_out_dir.exists():
            shutil.rmtree(region_out_dir)
        region_out_dir.mkdir(parents=True)
        if write_region_excel:
            log_step(log_path, "区域分析开始：已开启按区域导出 Excel，预计 %d 个文件。", n_potential_xlsx)
        else:
            log_step(log_path, "区域分析开始：已关闭按区域导出 Excel。")

        # 4. 按「区域编号、原输入行号」排序，保持轨迹原始时序。
        for f, (table, group) in enumerate(zip(all_tables, region_of_file_row)):
            rows = np.flatnonzero(group > 0)
            if not rows.size:
                log_step(log_path, "区域分析完成 D%d：没有有效 UWB 点，达标点数为 0。", f + 1)
                continue
            rows = rows[np.lexsort((rows, group[rows]))]
            cuts = np.flatnonzero(np.diff(group[rows]) != 0) + 1
            groups = np.split(rows, cuts)
            for p, point_rows in enumerate(groups, 1):
                rid = int(group[point_rows[0]]) - 1  # 数组位置 0-based，输出区域 ID 1-based
                subtable = table.iloc[point_rows].copy().reset_index(drop=True)
                n_pts = len(subtable)
                total_count[rid, f] = n_pts
                error = subtable[ERR_COL].to_numpy(dtype=float)
                finite_error = error[np.isfinite(error)]
                err_valid_count[rid, f] = int(finite_error.size)
                err_below_count[rid, f] = int(subtable["ErrorBelowThreshold"].sum())
                # 累加所有有效误差（不仅是误差小于阈值的点），不参与求和的 NaN/Inf 被跳过
                err_sum[rid, f] = float(finite_error.sum())
                for r, col_name in enumerate(RANGE_COLS):
                    vals = subtable[col_name].to_numpy(dtype=float)
                    valid_rate[rid, f, r] = np.count_nonzero(np.isfinite(vals)) / n_pts
                residual = subtable[RES_COL].to_numpy(dtype=float)
                finite_res = residual[np.isfinite(residual)]
                if finite_res.size:
                    res_mean[rid, f] = float(finite_res.mean())

                ux = subtable[UWB_X_COL].to_numpy(dtype=float)
                uy = subtable[UWB_Y_COL].to_numpy(dtype=float)
                distances = np.hypot(np.diff(ux), np.diff(uy))
                distances[~np.isfinite(distances)] = np.inf
                segment_starts = np.r_[0, np.flatnonzero(distances > path_break_threshold) + 1]
                segment_ends = np.r_[segment_starts[1:], n_pts]
                path_count[rid, f] = len(segment_starts)
                for r, col_name in enumerate(RANGE_COLS):
                    vals = subtable[col_name].to_numpy(dtype=float)
                    max_gap_ratio[rid, f, r] = max(
                        max_missing_ratio(vals[start:end])
                        for start, end in zip(segment_starts, segment_ends)
                    )
                obj_max_gap_ratio[rid, f] = max_gap_ratio[rid, f].max()

                if write_region_excel:
                    rid1 = rid + 1
                    region_dir = region_out_dir / f"Region{rid1:02d}"
                    region_dir.mkdir(parents=True, exist_ok=True)
                    out_path = region_dir / f"Region{rid1:02d}_D{f + 1}.xlsx"
                    subtable.to_excel(out_path, index=False, engine="openpyxl")
                    if p % 50 == 0:
                        log_step(log_path, "D%d：已导出 %d / %d 个区域 Excel 文件。", f + 1, p, len(groups))

            log_step(
                log_path,
                "区域分析完成 D%d：涉及 %d 个区域，误差小于 %.3f 米的点 %d 个，有效误差点 %d 个。",
                f + 1, len(groups), error_threshold,
                int(err_below_count[:, f].sum()), int(err_valid_count[:, f].sum()),
            )

        # 5. 每组区域统计及 D1~Dn 合并工作簿；保留达标点数，不导出达标率。
        combined_workbook = stat_out_dir / "All_D_ValidRate_Summary.xlsx"
        row_names = [f"Region{i:02d}" for i in range(1, n_regions + 1)]
        log_step(log_path, "开始导出：各 D 组区域统计表及合并工作簿。")
        with pd.ExcelWriter(combined_workbook, engine="openpyxl", mode="w") as combined_writer:
            for f in range(n_files):
                data = np.column_stack([
                    matlab_round(valid_rate[:, f, 0] * 100, 1),
                    matlab_round(valid_rate[:, f, 1] * 100, 1),
                    matlab_round(valid_rate[:, f, 2] * 100, 1),
                    matlab_round(obj_max_gap_ratio[:, f] * 100, 1),
                    matlab_round(res_mean[:, f], 4),
                    path_count[:, f], total_count[:, f],
                    np.full(n_regions, error_threshold),
                    err_below_count[:, f], err_valid_count[:, f],
                    err_sum[:, f],
                ])
                summary = pd.DataFrame(data, columns=STAT_COLS)
                summary.insert(0, "Row", row_names)  # MATLAB WriteRowNames=true
                per_d_path = stat_out_dir / f"D{f + 1}_ValidRate_AllRegions.xlsx"
                summary.to_excel(per_d_path, index=False, engine="openpyxl")
                summary.to_excel(combined_writer, sheet_name=f"D{f + 1}", index=False)
                log_step(log_path, "已导出 D%d 区域统计：达标点数 %d 个。", f + 1, int(err_below_count[:, f].sum()))

        # 6. 所有 D 组的误差点数直接相加，不转换成百分比。
        x_low = xmin + (region_rc[:, 1] - 1) * cell_size
        y_low = ymin + (region_rc[:, 0] - 1) * cell_size
        pooled_total = total_count.sum(axis=1)
        pooled_valid = err_valid_count.sum(axis=1)
        pooled_below = err_below_count.sum(axis=1)
        pooled = pd.DataFrame({
            "Region_ID": np.arange(1, n_regions + 1),
            "Grid_Row": region_rc[:, 0],
            "Grid_Col": region_rc[:, 1],
            "X_Min_m": x_low,
            "X_Max_m": x_low + cell_size,
            "Y_Min_m": y_low,
            "Y_Max_m": y_low + cell_size,
            "ErrorThreshold_m": error_threshold,
            "AllD_TotalCount": pooled_total,
            "AllD_ValidErrorCount": pooled_valid,
            "AllD_BelowThresholdCount": pooled_below,
        })
        for f in range(n_files):
            pooled[f"D{f + 1}_TotalCount"] = total_count[:, f]
            pooled[f"D{f + 1}_BelowThresholdCount"] = err_below_count[:, f]
        pooled_path = stat_out_dir / "All_D_Region_Error_Summary.xlsx"
        pooled.to_excel(pooled_path, index=False, engine="openpyxl")
        log_step(
            log_path,
            "已导出全部 D 组的区域误差汇总：总点数 %d 个，有效误差点 %d 个，误差小于 %.3f 米的点 %d 个。",
            int(pooled_total.sum()), int(pooled_valid.sum()), error_threshold, int(pooled_below.sum()),
        )
        # 7. 导出 MATLAB 原版的 UWB/RTK 轨迹图，与 Excel 统计共享相同的区域编号。
        image_path = None
        if make_trajectory_plot:
            log_step(log_path, "绘图：正在创建 UWB/RTK 轨迹及区域网格图。")
            image_path = draw_trajectory_map(
                all_tables, region_rc, pooled_total, pooled_below,
                xmin, xmax, ymin, ymax, n_cols, n_rows,
                cell_size, error_threshold, root_out_dir, log_path,
                show_region_grid=show_region_grid,
                show_region_labels=show_region_labels,
                show_error_count_on_map=show_error_count_on_map,
                max_grid_lines=max_grid_lines,
                max_region_labels=max_region_labels,
                max_draw_points_per_line=max_draw_points_per_line,
                plot_resolution=plot_resolution,
            )
        else:
            log_step(log_path, "绘图：已关闭，不生成轨迹 PNG。")
        log_step(log_path, "全部处理完成。")
        return {
            "region_count": n_regions,
            "region_excel_count": n_potential_xlsx if write_region_excel else 0,
            "total_points": int(pooled_total.sum()),
            "valid_error_points": int(pooled_valid.sum()),
            "below_threshold_points": int(pooled_below.sum()),
            "output_dir": root_out_dir,
            "plot_path": image_path,
        }
    except Exception as exc:
        log_step(log_path, "运行失败：%s", exc)
        log_step(log_path, "错误详细信息：\n%s", traceback.format_exc())
        raise


if __name__ == "__main__":
    run_analysis()


程序开始运行，固定输出目录：D:\Desktop\AllRegionAnalysisOutput
开始读取输入数据：共 5 个 Excel 数据组。
D1：从 D:\Desktop\missing_output\batch_output\D1_missing_uwb_rtk_aligned_result.xlsx 读取工作表「逐点结果」，共 1435 行。


Exception ignored in: <function ZipFile.__del__ at 0x00000207390EC940>
Traceback (most recent call last):
  File "c:\Users\30394\AppData\Local\Programs\Python\Python38\lib\zipfile.py", line 1821, in __del__
    self.close()
  File "c:\Users\30394\AppData\Local\Programs\Python\Python38\lib\zipfile.py", line 1838, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


D2：从 D:\Desktop\missing_output\batch_output\D2_missing_uwb_rtk_aligned_result.xlsx 读取工作表「逐点结果」，共 1315 行。
D3：从 D:\Desktop\missing_output\batch_output\D3_missing_uwb_rtk_aligned_result.xlsx 读取工作表「逐点结果」，共 1128 行。
D4：从 D:\Desktop\missing_output\batch_output\D4_uwb_rtk_aligned_result.xlsx 读取工作表「D4」，共 911 行。
D5：从 D:\Desktop\missing_output\batch_output\D5_uwb_rtk_aligned_result.xlsx 读取工作表「D5」，共 911 行。
网格范围：X 从 0.000 到 35.000 米，Y 从 -5.000 到 25.000 米。
实际包含数据的区域：29 个（候选区域共 42 个）。
区域分析开始：已开启按区域导出 Excel，预计 86 个文件。
区域分析完成 D1：涉及 21 个区域，误差小于 0.500 米的点 659 个，有效误差点 1435 个。
区域分析完成 D2：涉及 15 个区域，误差小于 0.500 米的点 917 个，有效误差点 1184 个。
区域分析完成 D3：涉及 12 个区域，误差小于 0.500 米的点 1113 个，有效误差点 1128 个。
区域分析完成 D4：涉及 19 个区域，误差小于 0.500 米的点 840 个，有效误差点 911 个。
区域分析完成 D5：涉及 19 个区域，误差小于 0.500 米的点 858 个，有效误差点 910 个。
开始导出：各 D 组区域统计表及合并工作簿。
已导出 D1 区域统计：达标点数 659 个。
已导出 D2 区域统计：达标点数 917 个。
已导出 D3 区域统计：达标点数 1113 个。
已导出 D4 区域统计：达标点数 840 个。
已导出 D5 区域统计：达标点数 858 个。
已导出全部 D 组的区域误差汇总：总点数 5700 个，有效误差点 5568 个，误差小于 0.500 米的点 4387 个。
绘图：正在创建 UW